# Feather v1 -- Kaggle Quickstart

Feather v1 lays hypervectors from a tropical (t-p-adic) feature lattice atop working-set
FFT ground; the whole thing runs **CPU-only**.

| claim | mechanism |
|---|---|
| 45-60 tok/s CPU | batched drafts `d(n)<=ceil(sqrt(n))+1, n in [0,64)` |
| (64,64) working set | token x dimension ground grid |
| final_output (1,64) | one-hot over the 64-token draft |
| 17.5x energy | tropical tag volume / linear flash term |
| 512x mem cheap | hypervector-dim projections, fp32 |
| 64x faster | rank-8 batched matmul fallback (AVX) |
| 10x fewer steps | K-FAC natural-gradient readout (training notebook) |

Run on a Kaggle notebook (or locally) after `pip install -e .`.

In [ ]:
!python -m feather_v1.hardware

In [ ]:
import json
import numpy as np
from feather_v1 import FeatherV1Model, FeatherV1Config
from feather_v1.hardware import kaggle_env, summary
from feather_v1.utils import byte_tokenize, byte_decode

env = kaggle_env()
print("is_kaggle:", env["is_kaggle"], "| run_type:", env["kernel_run_type"])
print("ram_gb:", env["ram_gb"], "| internet:", env["has_internet"], "| cuda:", env["cuda_devices"])

In [ ]:
# the packaged config (384-D, Kaggle-grade) for reference
def load_cfg():
    for cand in (
        "/kaggle/input/feather-v1-model/feather-v1-kaggle/config.json",
        "models/kaggle/config.json",
        "kaggle/configs/kaggle_cpu.json",
    ):
        try:
            with open(cand, encoding="utf-8") as fh:
                d = json.load(fh)
            if "feather_v1_config" in d:
                d = d["feather_v1_config"]
            return FeatherV1Config.from_dict(d)
        except OSError:
            continue
    return FeatherV1Config.auto()

print('packaged config:', load_cfg())

In [ ]:
# headline: 64-D working set, (64,64) grid -> (1,64) final_output
model = FeatherV1Model(
    FeatherV1Config(dim=64, seq_len=64, chunk_size=8, num_chunks=8, vocab_size=4096)
)
model.reset()
seq = np.random.default_rng(0).standard_normal((64, 64))
out = model.forward(seq)
assert out['final_output'].shape == (1, 64)  # one-hot over the auto-regressive draft
print('keys:', sorted(out))
print('final_output[0,:5]:', out['final_output'][0, :5])
print('token id:', out['token'])
print('kernel:', model.kernel)
print('cache_info:', out['cache_info'])

In [ ]:
# measured CPU throughput over repeated 64-token batches
import time
model.reset()
reps = 4
model.forward(seq); model.reset()
t0 = time.perf_counter()
for _ in range(reps):
    model.forward(seq)
elapsed = time.perf_counter() - t0
tokens = reps * 64
print(f'measured: {tokens / elapsed:.1f} tok/s over {tokens} tokens')

Next: `feather-v1-kaggle-benchmark.ipynb` (sustained numbers),
`feather-v1-kaggle-training.ipynb` (K-FAC readout),
`feather-v1-kaggle-inference.ipynb` (offline decode).